# End-to-End Credit Risk Analysis
## Notebook 1 — Data Preprocessing (Google Colab Version)
**Project:** Score Generation, Risk Classification & Approval Engine  
**Dataset:** Credit Risk Dataset (Kaggle — laotse/credit-risk-dataset)  
**Goal:** Load → Clean → Engineer Features → Save cleaned CSV to Google Drive

---
## Step 0 — Mount Google Drive
This keeps all your files saved permanently, even if the Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Create a project folder inside your Google Drive
project_path = '/content/drive/MyDrive/credit_risk_project'
data_path = f'{project_path}/data'

os.makedirs(data_path, exist_ok=True)
print('Project folder ready at:', project_path)

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

---
## Step 2 — Upload / Load Dataset
**Option A (recommended):** Upload the CSV once — it will save into Google Drive automatically for future sessions.  
**Option B:** If you already uploaded it earlier, skip Option A and just run the load cell below.

In [ ]:
# OPTION A — Run this only if the dataset is NOT already in your Drive folder
import os

if not os.path.exists(f'{data_path}/credit_risk_dataset.csv'):
    from google.colab import files
    print('Please choose credit_risk_dataset.csv from your computer...')
    uploaded = files.upload()

    # Move uploaded file into Drive project folder
    for filename in uploaded.keys():
        os.rename(filename, f'{data_path}/credit_risk_dataset.csv')
    print('File saved to Google Drive project folder!')
else:
    print('Dataset already exists in Drive — skipping upload.')

In [ ]:
# Load the dataset from Google Drive
df = pd.read_csv(f'{data_path}/credit_risk_dataset.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Basic info — column names, data types, non-null counts
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe()

---
## Step 3 — Inspect Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap')
plt.tight_layout()
plt.show()

---
## Step 4 — Handle Missing Values

In [ ]:
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
        print(f'Filled {col} with median: {df[col].median():.2f}')

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)
        print(f'Filled {col} with mode: {df[col].mode()[0]}')

print('\nMissing values after handling:', df.isnull().sum().sum())

---
## Step 5 — Explore the Data (EDA)

In [ ]:
plt.figure(figsize=(6, 4))
df['loan_status'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'])
plt.title('Loan Status Distribution (0 = No Default, 1 = Default)')
plt.xlabel('Loan Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('Default Rate:', round(df['loan_status'].mean() * 100, 2), '%')

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['person_income'], bins=50, kde=True, color='steelblue')
plt.title('Income Distribution')
plt.xlabel('Income')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x='loan_status', y='loan_amnt', data=df, palette=['steelblue', 'tomato'])
plt.title('Loan Amount by Default Status')
plt.xlabel('Default (0=No, 1=Yes)')
plt.ylabel('Loan Amount')
plt.tight_layout()
plt.show()

---
## Step 6 — Remove Outliers

In [ ]:
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(df)
    df = df[(df[column] >= lower) & (df[column] <= upper)]
    print(f'{column}: removed {before - len(df)} outlier rows')
    return df

df = remove_outliers_iqr(df, 'person_income')
df = remove_outliers_iqr(df, 'person_age')

print('\nDataset shape after outlier removal:', df.shape)

---
## Step 7 — Encode Categorical Variables

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', cat_cols)

for col in cat_cols:
    print(f'\n{col} unique values:', df[col].unique())

In [ ]:
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f'Encoded: {col}')

print('\nAll categorical columns encoded!')

---
## Step 8 — Feature Engineering (Derived Features)

In [ ]:
df['debt_to_income_ratio'] = df['loan_amnt'] / (df['person_income'] + 1)
df['loan_to_income_ratio'] = df['loan_amnt'] / (df['person_income'] + 1)
df['credit_utilization'] = df['loan_percent_income']
df['is_young'] = (df['person_age'] < 25).astype(int)
df['is_employed_long'] = (df['person_emp_length'] > 5).astype(int)

print('New features created:')
print(df[['debt_to_income_ratio', 'loan_to_income_ratio', 
          'credit_utilization', 'is_young', 'is_employed_long']].head())

---
## Step 9 — Normalize Numerical Features

In [ ]:
cols_to_scale = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'cb_person_cred_hist_length', 'debt_to_income_ratio',
    'loan_to_income_ratio', 'credit_utilization'
]

cols_to_scale = [col for col in cols_to_scale if col in df.columns]

scaler = MinMaxScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print('Scaled columns:', cols_to_scale)
print('\nSample after scaling:')
df[cols_to_scale].head()

---
## Step 10 — Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

print('\nTop correlations with loan_status:')
print(corr['loan_status'].sort_values(ascending=False))

---
## Step 11 — Final Check & Save to Google Drive

In [ ]:
print('Final Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nMissing Values:', df.isnull().sum().sum())

In [ ]:
# Save directly to Google Drive — no manual download needed
output_file = f'{data_path}/cleaned_data.csv'
df.to_csv(output_file, index=False)

import os
if os.path.exists(output_file):
    print('cleaned_data.csv saved successfully to Google Drive!')
    print('Location:', output_file)
    print('Total records:', len(df))
    print('Total features:', df.shape[1])
else:
    print('Save failed — check Drive permissions.')

print('\nDay 1 Complete! Ready for model training next.')

---
### Optional — Also download a local copy to your computer

In [ ]:
from google.colab import files
files.download(output_file)